## Active Learning experiment\n\nСравнение стратегий `entropy` vs `random` (кривые обучения).\nИспользует последний `artifacts/run_*/data_labeled_final.csv`.

In [ ]:
from pathlib import Path\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nfrom agents.al_agent import ActiveLearningAgent\n\nruns = sorted(Path("../artifacts").glob("run_*/data_labeled_final.csv"))\npath = runs[-1] if runs else None\ndf = pd.read_csv(path)\ndf.shape

In [ ]:
df = df.rename(columns={"label_final": "label"})\ndf = df[df["label"].astype(str).str.lower().ne("unknown")].copy()\ndf = df[df["text"].notna() & df["label"].notna()].copy()\ndf["label"].value_counts().head()

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])\nlabeled0 = train_df.sample(n=50, random_state=42)\npool = train_df.drop(index=labeled0.index)\n\nagent = ActiveLearningAgent()\nhist_entropy = agent.run_cycle(labeled0.reset_index(drop=True), pool.reset_index(drop=True), test_df.reset_index(drop=True), strategy="entropy", n_iterations=5, batch_size=20)\nhist_random = agent.run_cycle(labeled0.reset_index(drop=True), pool.reset_index(drop=True), test_df.reset_index(drop=True), strategy="random", n_iterations=5, batch_size=20)\n\nhistory = hist_entropy + hist_random\nhistory[:2]

In [ ]:
import matplotlib.pyplot as plt\nimport pandas as pd\n\nh = pd.DataFrame(history)\nplt.figure(figsize=(8,5))\nfor strat in h["strategy"].unique():\n    sub = h[h["strategy"]==strat].sort_values("n_labeled")\n    plt.plot(sub["n_labeled"], sub["f1"], marker="o", label=strat)\nplt.xlabel('n_labeled')\nplt.ylabel('F1 (macro)')\nplt.title('Active Learning: entropy vs random')\nplt.grid(True, alpha=0.3)\nplt.legend()\nplt.tight_layout()\nplt.show()